# レッスン18（フォローアップ）：<em>人間</em>が操作を承認したことを証明する領収書

このレッスンは、<strong>エージェント</strong> が行ったことと、<strong>ゲート</strong> が決定したことを証明します。このノートブックは、不足している半分を追加します：<strong>指定された人間</strong> が<strong>正確な</strong>操作を承認した証拠 — 完全な正規化された操作に対して人間が保持する別個の署名で、オフラインで検証されます。

ここにある両方の成果物は、<strong>レッスンの領収書と同じエンベロープ形状</strong> を使用しています：`type` フィールドを持つフラットなペイロードで、Ed25519 による署名を、正規化されたペイロードのバイト列の SHA-256 上で行い、構造化された `signature` オブジェクトを添付（署名バイトからは除外）しています。承認領収書は、新しい `type`（`human.approval.v1`）で、操作タイプと並びます。そのため、`verify_chain` はメインノートブックで構築したのと同じコードパスで、両方の成果物タイプをカバーします。これはまた、本レッスンが従うインターネットドラフト（draft-farley-acta-signed-receipts）の共同署名パスの形状でもあります。

メインノートブックのデモ検証器に対する意図的な改良点の一つとして、ここでの検証器は領収書内に含まれる公開鍵を信用するのではなく、`signature.key_id` を <strong>ピン留めされた鍵レジストリ</strong> に照合します。これは、レッスン自身のチェックリストが推奨する本番体制（「検証用公開鍵を公開する」）であり、これによって偽造は単なる持込鍵バイパスではなく拒否となります。

このノートブックで学ぶルール：**署名された承認は単独で権限を意味しません。** 権限は、承認領収書と操作領収書が実行時に同一の正規化操作に依然として紐付いており、ポリシーバージョン、鍵、有効期限が現在有効であり、かつ承認が既に消費されていない場合にのみ存在します。すべての失敗は<strong>異なる理由</strong>で拒否されるため、<em>権限が期限切れになった</em>のか、<em>実行された操作が変更された</em>のかを判別できます。


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## 正確なアクション

承認の単位は <strong>正準アクションオブジェクト</strong> であり、「返金を承認する」のような曖昧なラベルではなく、正確かつ完全に特定されたアクションです。オブジェクト全体に署名し（そこからダイジェストを導出することにより）、後で人間が<em>これ</em>だけを承認したことを証明できるようにします。


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## 1つのエンベロープ、2つの認証機関

すべてのレシートはレッスンのエンベロープです：`type` フィールドを持つフラットなペイロードと、署名済みバイトの一部ではない `signature` オブジェクト（`alg`、`sig`、`key_id`）を持ちます。`verify_envelope` は両方のレシート種に共通する構造＋署名の検証であり、どの <strong>ピン留めされたキー・レジストリ</strong> に対して `signature.key_id` を解決するかが権限を分けています：

- <strong>承認レシート</strong>（`human.approval.v1`）— 指名された承認者、完全な正規化されたアクション <strong>およびそのダイジェスト</strong>、`policy_version`、発行＋期限切れのタイムスタンプ。チェーンレベルで一度限りの消費を追跡します。
- <strong>アクションレシート</strong>（`agent.action.v1`）— エージェントのアイデンティティ、`run_id`、同じ正規化されたアクションの <strong>ダイジェスト</strong>、実行結果＋タイムスタンプ、`parent_approval_ref`：承認の `receipt_hash`、レッスンのチェーンの `previous_receipt_hash` と同じ慣習です。

共通の `action_digest` フィールドが結合時のバインディングの依存対象です。`key_id` は単に検索のヒントとして署名オブジェクトに存在し、これを異なるピン留めキーに向け直すと署名検証が失敗するため、何の権限も与えません。


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: 実際にバインディングが決定される場所

`verify_chain` は、2つの署名検証の便利なラッパーではありません。共有された正準 `action_digest`、承認のポリシー/鍵/有効期限の<strong>新鮮さ</strong>、そして承認の<strong>一度きりの消費</strong>が、一緒に、<em>まさに今</em>実行されているアクションに対して検証される唯一の場所です。

それぞれの失敗は<strong>異なる理由</strong>で拒否されるため、拒否を受け取った読者は権限が古くなったのか（ポリシーが変更された、鍵が回転された、承認が期限切れ、承認が消費された）、それともまだ有効な承認のもとで実行されたアクションが変更されたのか（ダイジェスト差し替え）を判別できます。


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## バインディングが検出するもの

以下の各ケースはすべて、<strong>異なる理由</strong>で<strong>クローズド</strong>に失敗します。最初のブロックは古典的なセット（改ざん、混乱した代理、リプレイ、いずれかの権限による偽造、不正な入力）です。二番目のブロックは、性質を単なる主張ではなく現実のものにするペアです：

- <strong>古くなった権限</strong> — 署名はまだ有効ですが、ポリシーバージョンが移動された、承認者のキーが固定レジストリから回転された、または実行前に承認が期限切れとなった場合；
- <strong>ダイジェストの置き換え</strong> — `parent_approval_ref` が <em>実際の</em> 承認を指している有効に署名されたアクションレシートですが、その承認の標準的なアクションダイジェストが実際に実行されるアクションと一致しない場合。


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## これが証明すること — そして証明しないこと

**証明すること:** 名前のある人間が <em>この正確な正規アクション</em>（完全なアクション＋ダイジェスト、ピン留めされたレジストリから解決されたキーで署名）を承認し、エージェントが <em>正確にその承認されたアクション</em>（同じダイジェスト、承認に `receipt_hash` で結び付けられたレシート、レッスンの独自チェーン規約）を実行したこと — 承認のポリシーバージョン、キー、有効期限がまだ有効な間に、ちょうど一度だけ。どちらかが変わるとチェーンは閉じて拒否され、その拒否理由は <strong>どの</strong> プロパティが壊れたかを示す：期限切れの権限対変わったアクション。

**証明しないこと:** 承認UIが人間に自分が署名している内容を表示したかどうか（WYSIWYS は別の問題）、キーがローテーション前に強制または盗まれていなかったか、または下流の結果がアクションに一致したかどうか。署名は ≠ 承認：古いポリシー、ローテーションされたキー、期限切れの時間枠、または異なるダイジェストに対する有効な署名はここでは何も意味しない。

この2種類のレシートは、レッスンのエンベロープと1つの `verify_chain` コードパスを共有しているのは意図的である：メインノートブックでアクションレシート用に作ったバインディングは、人間の承認を検証する同じコードである。1つの検証契約、別々のピン留めされた権限、正規アクションダイジェストで結ばれ、他は何もない。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
